In [1]:
!pip install numpy pandas jupyter-black scikit-learn xgboost matplotlib requests lightgbm

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

%load_ext jupyter_black

In [68]:
df = pd.read_csv("resale2017-2025.csv")

In [69]:
df["month"] = pd.to_datetime(df["month"], format="%Y-%m")

df["month_num"] = df["month"].dt.month
df["year"] = df["month"].dt.year

df["month_sin"] = np.sin((df["month_num"] - 1) * (2 * np.pi / 12))
df["month_cos"] = np.cos((df["month_num"] - 1) * (2 * np.pi / 12))

df["salesYear_fr_2017"] = df["year"] - 2017

In [70]:
region_map = {
    "SENGKANG": "North-East",
    "PUNGGOL": "North-East",
    "HOUGANG": "North-East",
    "WOODLANDS": "North",
    "YISHUN": "North",
    "SEMBAWANG": "North",
    "TAMPINES": "East",
    "PASIR RIS": "East",
    "BEDOK": "East",
    "GEYLANG": "East",
    "MARINE PARADE": "East",
    "JURONG WEST": "West",
    "JURONG EAST": "West",
    "CHOA CHU KANG": "West",
    "BUKIT BATOK": "West",
    "BUKIT PANJANG": "West",
    "CLEMENTI": "West",
    "ANG MO KIO": "Central",
    "BISHAN": "Central",
    "TOA PAYOH": "Central",
    "KALLANG/WHAMPOA": "Central",
    "QUEENSTOWN": "Central",
    "BUKIT MERAH": "Central",
    "SERANGOON": "Central",
    "CENTRAL AREA": "Central",
    "BUKIT TIMAH": "Central",
}

In [71]:
df["region"] = df["town"].map(region_map)

In [72]:
gdp = pd.read_csv("gdp.csv")
gdp.rename(
    columns={"Year": "year", "GDP At Current Market Prices": "GDP"}, inplace=True
)
gdp2025 = pd.DataFrame([{"year": 2025, "GDP": 731436.1}])
gdp = pd.concat([gdp, gdp2025], ignore_index=True)
gdp.sort_values(by="year", ascending=False)

,year,GDP
35,2025,731436.1
0,2024,731436.1
1,2023,678687.5
2,2022,701766.1
3,2021,586553.1
4,2020,481758.8
5,2019,513144.4
6,2018,508680.3
7,2017,474587.1
8,2016,441606.3


In [73]:
gdp = gdp[gdp["year"] >= 2017]

In [74]:
df = df.merge(gdp, on="year", how="left")

In [75]:
df["add"] = df["block"] + " " + df["street_name"]

In [76]:
import requests

add = df["add"].loc[0]

params = {
    "searchVal": add,
    "returnGeom": "Y",
    "getAddrDetails": "N",
    "pageNum": 1,
}

url = "https://www.onemap.gov.sg/api/common/elastic/search"
response = requests.get(url, params=params)
data = response.json()

result = data["results"][0]
print(result["LATITUDE"])
print(result["LONGITUDE"])

1.36200453938712
103.853879910407


In [12]:
import requests

def coord(add):
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {"searchVal": add, "returnGeom": "Y", "getAddrDetails": "N", "pageNum": 1}
    response = requests.get(url, params=params)
    data = response.json()
    result = data["results"][0]
    return result["LATITUDE"], result["LONGITUDE"]

In [77]:
add = df["add"]

In [78]:
coords = pd.read_csv("coords_with_walk_metrics.csv")

In [17]:
coords.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9902 entries, 0 to 9901
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   add                      9902 non-null   object 
 1   latitude                 9902 non-null   float64
 2   longitude                9902 non-null   float64
 3   nearest_sch_index        9902 non-null   int64  
 4   distance_to_nearest_sch  9902 non-null   float64
 5   nearest_sch_lat          9902 non-null   float64
 6   nearest_sch_lon          9902 non-null   float64
 7   nearest_sch_name         9902 non-null   object 
 8   nearest_mrt_index        9902 non-null   int64  
 9   distance_to_nearest_mrt  9902 non-null   float64
 10  nearest_mrt_lat          9902 non-null   float64
 11  nearest_mrt_lon          9902 non-null   float64
 12  nearest_mrt_name         9902 non-null   object 
 13  walk_time_to_school      7950 non-null   float64
 14  walk_distance_to_school 

In [79]:
df["add"] = df["block"] + " " + df["street_name"]

In [80]:
df["storey"] = df["storey_range"].str.split(" ", n=1, expand=True)[0]

In [81]:
df = df.merge(coords, on="add", how="left")

In [82]:
df["remaining_lease_year"] = df["remaining_lease"].str.split(" ", n=1, expand=True)[0]

In [83]:
df["remaining_lease_year"] = df["remaining_lease_year"].astype(int)

In [67]:
df.columns

Index(['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range',
       'floor_area_sqm', 'flat_model', 'lease_commence_date',
       'remaining_lease', 'resale_price', 'month_num', 'year', 'month_sin',
       'month_cos', 'salesYear_fr_2017', 'region', 'GDP', 'add', 'latitude_x',
       'longitude_x', 'nearest_sch_index_x', 'distance_to_nearest_sch_x',
       'nearest_sch_lat_x', 'nearest_sch_lon_x', 'nearest_sch_name_x',
       'nearest_mrt_index_x', 'distance_to_nearest_mrt_x', 'nearest_mrt_lat_x',
       'nearest_mrt_lon_x', 'nearest_mrt_name_x', 'walk_time_to_school_x',
       'walk_distance_to_school_x', 'walk_time_to_mrt_x',
       'walk_distance_to_mrt_x', 'storey', 'remaining_lease_year',
       'latitude_y', 'longitude_y', 'nearest_sch_index_y',
       'distance_to_nearest_sch_y', 'nearest_sch_lat_y', 'nearest_sch_lon_y',
       'nearest_sch_name_y', 'nearest_mrt_index_y',
       'distance_to_nearest_mrt_y', 'nearest_mrt_lat_y', 'nearest_mrt_lon_y',
       'neares

In [87]:
df_test = df[df["month"] > "2024-06"]
df_train = df[df["month"] <= "2024-06"]

In [88]:
df_train.columns

Index(['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range',
       'floor_area_sqm', 'flat_model', 'lease_commence_date',
       'remaining_lease', 'resale_price', 'month_num', 'year', 'month_sin',
       'month_cos', 'salesYear_fr_2017', 'region', 'GDP', 'add', 'storey',
       'latitude', 'longitude', 'nearest_sch_index', 'distance_to_nearest_sch',
       'nearest_sch_lat', 'nearest_sch_lon', 'nearest_sch_name',
       'nearest_mrt_index', 'distance_to_nearest_mrt', 'nearest_mrt_lat',
       'nearest_mrt_lon', 'nearest_mrt_name', 'walk_time_to_school',
       'walk_distance_to_school', 'walk_time_to_mrt', 'walk_distance_to_mrt',
       'remaining_lease_year'],
      dtype='object')

In [89]:
# Preprocessor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

categorical_cols = ["town", "flat_type", "flat_model", "region", "storey"]
scaled_numeric_cols = [
    "floor_area_sqm",
    "GDP",
    "remaining_lease_year",
    "walk_distance_to_school",
    "walk_distance_to_mrt",
]
date_cols = ["month_sin", "month_cos", "salesYear_fr_2017"]

scaled_num = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler())]
)

preprocessor = ColumnTransformer(
    [
        ("scaled_num", scaled_num, scaled_numeric_cols),
        ("keep_scaled", "passthrough", scaled_numeric_cols),
        ("keep_dates", "passthrough", date_cols),
    ]
)

In [97]:
from lightgbm import LGBMRegressor
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

for col in categorical_cols:
    df[col] = df[col].astype("category")

X = df_train[categorical_cols + scaled_numeric_cols + date_cols]
y = df_train["resale_price"]

tscv = TimeSeriesSplit(n_splits=5)
fold_maes = []
fold = 1

for train_index, val_index in tscv.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = LGBMRegressor(
        max_depth=10,
        n_estimators=200,
        learning_rate=0.1,
        objective="regression",
        random_state=42,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="mae",
        callbacks=[lgb.early_stopping(5)],
    )

    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    fold_maes.append(mae)

np.mean(fold_maes)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000636 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 811
[LightGBM] [Info] Number of data points in the train set: 30496, number of used features: 13
[LightGBM] [Info] Start training from score 445235.752636
Training until validation scores don't improve for 5 rounds
Did not meet early stopping. Best iteration is:
[200]	valid_0's l1: 25240.2	valid_0's l2: 1.22754e+09
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000861 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 817
[LightGBM] [Info] Number of data points in the train set: 60991, number of used features: 13
[LightGBM] [Info] Start training from score 439406.165674
Training until val

np.float64(32481.43550452534)

In [92]:
model = LGBMRegressor(
    max_depth=10,
    n_estimators=200,
    learning_rate=0.1,
    objective="regression",
    random_state=42
)

pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])

In [93]:
X_test = df_test[categorical_cols + scaled_numeric_cols + date_cols]

for col in categorical_cols:
    X_test[col] = X_test[col].astype("category")

/var/folders/0v/s2ctb70n01q4rpp9lkjdk5c40000gn/T/ipykernel_57050/910418592.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[col] = X_test[col].astype("category")


In [94]:
pipeline.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002233 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1497
[LightGBM] [Info] Number of data points in the train set: 152476, number of used features: 13
[LightGBM] [Info] Start training from score 482610.262208


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('scaled_num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['floor_area_sqm', 'GDP',
                                                   'remaining_lease_year',
                                                   'walk_distance_to_school',
                                                   'walk_distance_to_mrt']),
                                                 ('keep_scaled', 'passthrough',
                                                  ['floor_area_sqm', 'GDP',
                                                   'remaining_lease_year',
                                                   'walk_distance_to_school',
                                                   'walk_distance_to_mrt']),
                                                 ('keep_dates', 'passthrough',
                                                  ['month_sin', 'month_cos',
                                                   'salesYear_fr_2017'])])),
                ('model',
                 LGBMRegressor(max_depth=10, n_estimators=200,
                               objective='regression', random_state=42))])

In [95]:
y_test_pred = pipeline.predict(X_test)

/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_chat_bot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [96]:
y_test = df_test["resale_price"]
mae = mean_absolute_error(y_test, y_test_pred)
mae

74827.17134155158

In [98]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", LGBMRegressor(objective="regression", random_state=42)),
    ]
)

param_grid = {
    "model__max_depth": [5, 10, 15],
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__num_leaves": [20, 31, 50],
    "model__min_child_samples": [20, 50],
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=2,
)

for col in categorical_cols:
    df_train[col] = df_train[col].astype("category")

X = df_train[categorical_cols + scaled_numeric_cols + date_cols]
y = df_train["resale_price"]

grid_search.fit(X, y)

Fitting 5 folds for each of 162 candidates, totalling 810 fits


/var/folders/0v/s2ctb70n01q4rpp9lkjdk5c40000gn/T/ipykernel_57050/3738918715.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train[col] = df_train[col].astype("category")
/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_chat_bot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_chat_bot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_chat_bot/lib/python3.10/site-packages/sklearn/utils/va

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.060299 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1442
[LightGBM] [Info] Number of data points in the train set: 30496, number of used features: 13
[LightGBM] [Info] Start training from score 445235.752636
[CV] END model__learning_rate=0.01, model__max_depth=5, model__min_child_samples=20, model__n_estimators=100, model__num_leaves=31; total time=   3.0s
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002302 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1473
[LightGBM] [Info] Number of data points in the train set: 91486, number of used features: 13
[LightGBM] [Info] Start training from score 444525.182298
[LightGBM] [Warning] No fur

/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_chat_bot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_chat_bot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_chat_bot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006930 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1473
[LightGBM] [Info] Number of data points in the train set: 91486, number of used features: 13
[LightGBM] [Info] Start training from score 444525.182298
[CV] END model__learning_rate=0.01, model__max_depth=5, model__min_child_samples=20, model__n_estimators=100, model__num_leaves=20; total time=   2.8s
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004076 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1453
[LightGBM] [Info] Number of data points in the train set: 60991, number of used features: 13
[LightGBM] [Info] Start training from score 439406.165674
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Ligh

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_chat_bot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_chat_bot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_chat_bot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_chat_bot/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/fel/.pyenv/versions/3.10.6/envs/hdb_resale_ch

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002575 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1504
[LightGBM] [Info] Number of data points in the train set: 182971, number of used features: 13
[LightGBM] [Info] Start training from score 499022.994095


GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('scaled_num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          RobustScaler())]),
                                                                         ['floor_area_sqm',
                                                                          'GDP',
                                                                          'remaining_lease_year',
                                                                          'walk_distance_to_school',
                                                                          'walk_distance_...
                                                                         ['month_sin',
                                                                          'month_cos',
                                                                          'salesYear_fr_2017'])])),
                                       ('model',
                                        LGBMRegressor(objective='regression',
                                                      random_state=42))]),
             n_jobs=-1,
             param_grid={'model__learning_rate': [0.01, 0.05, 0.1],
                         'model__max_depth': [5, 10, 15],
                         'model__min_child_samples': [20, 50],
                         'model__n_estimators': [100, 200, 300],
                         'model__num_leaves': [20, 31, 50]},
             scoring='neg_mean_absolute_error', verbose=2)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002502 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1490
[LightGBM] [Info] Number of data points in the train set: 121981, number of used features: 13
[LightGBM] [Info] Start training from score 463677.882533
[CV] END model__learning_rate=0.1, model__max_depth=10, model__min_child_samples=20, model__n_estimators=100, model__num_leaves=50; total time=   5.1s
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006805 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1473
[LightGBM] [Info] Number of data points in the train set: 91486, number of used features: 13
[LightGBM] [Info] Start training from score 444525.182298
[CV] END model__learning_rate=0.1, model__max_depth=10, model__min_child_samples=20, model

In [ ]:
print("Best MAE:", -grid_search.best_score_)
print("Best parameters:", grid_search.best_params_)
